# Driver Intent — Round 2 (target: >75%)

Round 1 reached 65.35% mean / 67.5% best fold. Round 2 changes:
- **Conservative cleanup**: protect `normal_forward` and `pedestrian_monitor` from relabeling (keeps original 500/500/500/500 balance).
- **Bigger model**: hidden 192 (was 128).
- **Denser windows**: stride 5 (was 10) → ~2× training samples.
- **Longer training**: 80 epochs, patience 25 (was 40 / 12).
- **Keep per-fold checkpoints** so we can ensemble.
- **5-fold ensemble at eval** for the final number.
- **Class-balanced sampling** (`--balance-classes`): per-class WeightedRandomSampler so each batch has equal class representation regardless of raw frequency.
- **Low-confidence weak labels dropped** (`--min-weak-conf 0.6`): sequences whose `weak_label_meta.consensus_confidence` is below 0.6 are skipped at load time.

**Total Colab time:** ~3 hrs.

**Before running:** upload `data/hdd_cleaned_v2_validated.json` to `MyDrive/intent_data/`.

## 1. Setup — mount Drive, clone repo, install deps

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!rm -rf driver_intent_monitoring_system
!git clone https://github.com/AranD3V/driver_intent_monitoring_system.git
%cd driver_intent_monitoring_system

In [ ]:
!pip install -q -r requirements-colab.txt

In [ ]:
!mkdir -p /content/drive/MyDrive/intent_data /content/drive/MyDrive/intent_models /content/drive/MyDrive/intent_reports
!rm -rf data models reports
!ln -s /content/drive/MyDrive/intent_data    data
!ln -s /content/drive/MyDrive/intent_models  models
!ln -s /content/drive/MyDrive/intent_reports reports
!ls -la data/

In [ ]:
import torch, json, os
print('CUDA available :', torch.cuda.is_available())
print('GPU            :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
for f in ['data/hdd_train.json', 'data/hdd_cleaned_v2_validated.json']:
    if os.path.exists(f):
        d = json.load(open(f))
        print(f'{f}: {len(d)} sequences')
    else:
        print(f'MISSING: {f}')

## 2. Round 2 training — bigger model, denser windows, keep folds

In [ ]:
!python scripts/train_intent.py train-kfold \
    --data data/hdd_cleaned_v2_validated.json \
    --output models/weak_v2.pth \
    --epochs 80 --batch 32 --seq-len 50 --stride 5 \
    --hidden 192 --dropout 0.3 --lr 3e-4 \
    --early-stop 25 --seed 42 --overwrite \
    --weak-aug --use-weak-weights --keep-folds \
    --balance-classes --min-weak-conf 0.6

## 3. Single-checkpoint eval (best fold)

In [ ]:
!python scripts/train_intent.py evaluate \
    --output models/weak_v2.pth --data data/hdd_cleaned_v2_validated.json \
    --batch 32 --seq-len 50 --otc | tee reports/eval_weak_v2.txt

## 4. 5-fold ensemble eval (the headline number)

In [ ]:
import glob
fold_paths = sorted(glob.glob('models/weak_v2_fold*.pth'))
print(f'Found {len(fold_paths)} fold checkpoints:')
for p in fold_paths: print(' ', p)

In [ ]:
!python scripts/train_intent.py ensemble \
    --models models/weak_v2_fold0.pth models/weak_v2_fold1.pth \
             models/weak_v2_fold2.pth models/weak_v2_fold3.pth \
             models/weak_v2_fold4.pth \
    --data data/hdd_cleaned_v2_validated.json \
    --batch 32 --seq-len 50 --otc | tee reports/eval_weak_v2_ensemble.txt

## 5. Pick canonical + summary table

In [ ]:
import re, shutil, json

def parse_acc(path):
    if not os.path.exists(path): return -1.0
    txt = open(path).read()
    m = re.search(r'accuracy\s+([\d.]+)', txt)
    return float(m.group(1)) * 100 if m else -1.0

kfold_summary = json.load(open('models/kfold_summary.json'))
mean_kfold = kfold_summary['mean_val_acc']
std_kfold  = kfold_summary['std_val_acc']
best_fold  = max(kfold_summary['fold_accs'])

acc_single   = parse_acc('reports/eval_weak_v2.txt')
acc_ensemble = parse_acc('reports/eval_weak_v2_ensemble.txt')

print('='*60)
print(f'{"Metric":<35}{"Acc":>10}')
print('='*60)
print(f'{"Round 1 mean kfold":<35}{65.35:>10.2f}')
print(f'{"Round 1 best fold":<35}{67.50:>10.2f}')
print('-'*60)
print(f'{"Round 2 mean kfold":<35}{mean_kfold:>10.2f}')
print(f'{"Round 2 best fold":<35}{best_fold:>10.2f}')
print(f'{"Round 2 single-best eval":<35}{acc_single:>10.2f}')
print(f'{"Round 2 5-fold ensemble eval":<35}{acc_ensemble:>10.2f}')
print('='*60)
print('\nPer-class F1 (round 2 kfold mean):')
for c, v in kfold_summary['per_class_f1'].items():
    print(f'  {c:<25} {v["mean"]:.3f} +/- {v["std"]:.3f}')

# Save winner as canonical
winner_acc = max(acc_ensemble, best_fold)
if acc_ensemble >= best_fold:
    print(f'\nCanonical = ensemble  ({acc_ensemble:.2f}%)')
else:
    shutil.copy('models/weak_v2.pth', 'models/intent_canonical.pth')
    print(f'\nCanonical -> models/intent_canonical.pth (single best fold {best_fold:.2f}%)')

In [ ]:
import shutil

acc_v2 = parse_acc('reports/eval_weak_v2_ensemble.txt')
acc_v3 = parse_acc('reports/eval_weak_v3_ensemble.txt')

print('='*60)
print(f'{"Variant":<35}{"Ensemble Acc":>15}')
print('='*60)
print(f'{"Round 2 (min-weak-conf 0.6)":<35}{acc_v2:>15.2f}')
print(f'{"Round 3 (min-weak-conf 0.4)":<35}{acc_v3:>15.2f}')
print('='*60)

if acc_v3 > acc_v2:
    shutil.copy('models/weak_v3.pth', 'models/intent_canonical.pth')
    print(f'\nWinner: Round 3 (+{acc_v3 - acc_v2:.2f} pts) -> models/intent_canonical.pth')
else:
    shutil.copy('models/weak_v2.pth', 'models/intent_canonical.pth')
    print(f'\nWinner: Round 2 (Round 3 not better by {acc_v2 - acc_v3:.2f} pts) -> models/intent_canonical.pth')
    print('Keeping Round 2\'s 73% as the final result.')

In [ ]:
!python scripts/train_intent.py ensemble \
    --models models/weak_v3_fold0.pth models/weak_v3_fold1.pth \
             models/weak_v3_fold2.pth models/weak_v3_fold3.pth \
             models/weak_v3_fold4.pth \
    --data data/hdd_cleaned_v2_validated.json \
    --batch 32 --seq-len 50 --otc | tee reports/eval_weak_v3_ensemble.txt

In [ ]:
!python scripts/train_intent.py train-kfold \
    --data data/hdd_cleaned_v2_validated.json \
    --output models/weak_v3.pth \
    --epochs 80 --batch 32 --seq-len 50 --stride 5 \
    --hidden 192 --dropout 0.3 --lr 3e-4 \
    --early-stop 25 --seed 42 --overwrite \
    --weak-aug --use-weak-weights --keep-folds \
    --balance-classes --min-weak-conf 0.4

## 6. Round 3 (optional) — relax min-weak-conf 0.6 → 0.4

Round 2 hit 73% ensemble (target was >75%). Theory: `--min-weak-conf 0.6` is dropping too many sequences, starving each fold of data. Round 3 keeps every other flag the same and only relaxes the confidence floor to 0.4.

If Round 3 ensemble ≥ Round 2 ensemble, copy `weak_v3.pth` as canonical. Otherwise keep Round 2's 73%.